[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/01_tag_1_machine_learning_playground.ipynb)

# Tag 1 Machine Learning Playground

Dieses Notebook begleitet Tag 1 praktisch. Es zeigt Regression, Klassifikation, Clustering, Metriken, Train/Valid/Test-Splits und den Bias-Varianz-Tradeoff mit kleinen Datensätzen ohne Internet-Downloads.

## 1. Setup und Grundbegriffe

- **Sample**: eine Beobachtung, also eine Zeile in der Datentabelle.
- **Feature**: eine Eingabevariable, also eine Spalte in `X`.
- **Label/Zielgröße**: die gesuchte Ausgabe `y`.
- **Modellparameter**: Werte, die das Modell beim Training lernt.
- **Hyperparameter**: Einstellungen, die wir vor dem Training wählen, z. B. Baumtiefe oder Anzahl Nachbarn.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

from sklearn.datasets import make_regression, make_classification, make_blobs, load_diabetes, load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, median_absolute_error,
    mean_absolute_percentage_error, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, silhouette_score, adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (7, 4)
print('Setup abgeschlossen.')

In [ ]:
# Kleiner synthetischer Datensatz: X enthält Features, y die Zielgröße.
X_demo, y_demo = make_regression(n_samples=80, n_features=2, noise=12, random_state=RANDOM_STATE)
demo_df = pd.DataFrame(X_demo, columns=['Feature 1', 'Feature 2'])
demo_df['Zielgröße y'] = y_demo

display(demo_df.head())
print('X-Form:', X_demo.shape, '| y-Form:', y_demo.shape)

plt.scatter(demo_df['Feature 1'], demo_df['Zielgröße y'], c=demo_df['Feature 2'], cmap='viridis')
plt.colorbar(label='Feature 2')
plt.xlabel('Feature 1')
plt.ylabel('Zielgröße y')
plt.title('Samples, Features und Zielgröße')
plt.show()

## 2. Überwachtes Lernen als Regression

Regression sagt numerische Werte voraus. Wir nutzen `load_diabetes` und zusätzlich einen kleinen synthetischen Mietpreis-Datensatz.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

models_reg = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'DecisionTreeRegressor': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_true, y_pred),
        'MedianAE': median_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
    }

rows = []
for name, model in models_reg.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    row = {'Modell': name, 'CV_MAE': -cv.mean()}
    row.update(regression_metrics(y_test, pred))
    rows.append(row)
reg_results = pd.DataFrame(rows).sort_values('RMSE')
display(reg_results)

**Metriken kurz erklärt**

- **MAE**: durchschnittlicher absoluter Fehler, gut interpretierbar und robuster gegenüber Ausreißern.
- **MSE/RMSE**: quadriert Fehler; große Fehler fallen stärker auf. RMSE hat wieder die Einheit der Zielgröße.
- **R²**: Anteil erklärter Varianz; höher ist besser.
- **Median Absolute Error**: sehr robust, da der Median genutzt wird.
- **MAPE**: relativer Fehler in Prozentnähe; problematisch bei Zielwerten nahe 0.

In [ ]:
# Synthetischer Mietpreis-Datensatz
rng = np.random.default_rng(RANDOM_STATE)
wohnflaeche = rng.normal(75, 18, 120).clip(25, 150)
zimmer = rng.integers(1, 6, 120)
entfernung = rng.uniform(0.5, 18, 120)
miete = 8.5 * wohnflaeche + 120 * zimmer - 18 * entfernung + rng.normal(0, 90, 120)
rent_df = pd.DataFrame({'Wohnfläche': wohnflaeche, 'Zimmer': zimmer, 'Entfernung Zentrum': entfernung, 'Miete': miete})
display(rent_df.head())

plt.scatter(rent_df['Wohnfläche'], rent_df['Miete'], alpha=0.75)
plt.xlabel('Wohnfläche')
plt.ylabel('Miete')
plt.title('Synthetische Mietpreise')
plt.show()

In [ ]:
best_reg = models_reg['RandomForestRegressor']
y_pred = best_reg.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, y_pred, alpha=0.75)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('y true')
axes[0].set_ylabel('y pred')
axes[0].set_title('Vorhersage gegen Wahrheit')
residuen = y_test - y_pred
axes[1].scatter(y_pred, residuen, alpha=0.75)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Vorhersage')
axes[1].set_ylabel('Residuum')
axes[1].set_title('Residuenplot')
plt.tight_layout(); plt.show()

In [ ]:
def outlier_demo(anzahl_ausreisser=2, staerke=80):
    y_true = np.ones(40) * 100
    y_pred = y_true + np.random.default_rng(RANDOM_STATE).normal(0, 5, size=40)
    y_pred[:anzahl_ausreisser] += staerke
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    plt.scatter(range(len(y_true)), y_pred - y_true, alpha=0.8)
    plt.axhline(0, color='black', linewidth=1)
    plt.title(f'Outlier-Demo: MAE={mae:.2f}, RMSE={rmse:.2f}')
    plt.xlabel('Sample')
    plt.ylabel('Fehler')
    plt.show()

widgets.interact(outlier_demo, anzahl_ausreisser=widgets.IntSlider(2, 0, 10), staerke=widgets.IntSlider(80, 0, 250, 10));

### Aufgaben

1. Ändere `max_depth` beim Decision Tree und beobachte RMSE und R².
2. Welche Metrik würdest du für Mietpreise mit wenigen extrem teuren Wohnungen bevorzugen? Begründe kurz.

## 3. Überwachtes Lernen als Klassifikation

Klassifikation sagt Klassen voraus, zum Beispiel krank/gesund oder Ziffer 0 bis 9.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=RANDOM_STATE)

models_clf = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    'KNeighborsClassifier': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}
rows=[]
for name, model in models_clf.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    proba = model.predict_proba(Xc_test)[:, 1]
    rows.append({'Modell': name, 'Accuracy': accuracy_score(yc_test, pred), 'Balanced Accuracy': balanced_accuracy_score(yc_test, pred),
                 'Precision': precision_score(yc_test, pred), 'Recall': recall_score(yc_test, pred), 'F1': f1_score(yc_test, pred),
                 'ROC AUC': roc_auc_score(yc_test, proba), 'PR AUC': average_precision_score(yc_test, proba), 'MCC': matthews_corrcoef(yc_test, pred)})
display(pd.DataFrame(rows).sort_values('F1', ascending=False))

logreg = models_clf['LogisticRegression']
yc_pred = logreg.predict(Xc_test)
ConfusionMatrixDisplay.from_predictions(yc_test, yc_pred, display_labels=cancer.target_names)
plt.title('Confusion Matrix')
plt.show()
print(classification_report(yc_test, yc_pred, target_names=cancer.target_names))

In [ ]:
# Digits kurz laden und visualisieren: Mehrklassen-Klassifikation.
digits = load_digits()
fig, axes = plt.subplots(1, 6, figsize=(8, 2))
for ax, image, label in zip(axes, digits.images[:6], digits.target[:6]):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Label {label}')
    ax.axis('off')
plt.suptitle('load_digits: kleine 8x8-Bilder')
plt.show()

In [ ]:
# Imbalanced Data Demo: Viele Nullen, wenige Einsen.
Xi, yi = make_classification(n_samples=1200, n_features=8, n_informative=3, weights=[0.94, 0.06], flip_y=0.02, random_state=RANDOM_STATE)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=RANDOM_STATE)
majority_pred = np.zeros_like(yi_test)
print('Nur Mehrheitsklasse vorhersagen')
print('Accuracy:', accuracy_score(yi_test, majority_pred).round(3))
print('Recall positive Klasse:', recall_score(yi_test, majority_pred, zero_division=0).round(3))
print('Balanced Accuracy:', balanced_accuracy_score(yi_test, majority_pred).round(3))

In [ ]:
proba = logreg.predict_proba(Xc_test)[:, 1]
def threshold_demo(threshold=0.50):
    pred = (proba >= threshold).astype(int)
    fp = confusion_matrix(yc_test, pred)[0, 1]
    fn = confusion_matrix(yc_test, pred)[1, 0]
    print(f'Precision: {precision_score(yc_test, pred):.3f}')
    print(f'Recall:    {recall_score(yc_test, pred):.3f}')
    print(f'F1:        {f1_score(yc_test, pred):.3f}')
    print(f'False Positives: {fp} | False Negatives: {fn}')
    ConfusionMatrixDisplay.from_predictions(yc_test, pred, display_labels=cancer.target_names)
    plt.show()
widgets.interact(threshold_demo, threshold=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05));

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(yc_test, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(yc_test, proba, ax=axes[1])
axes[0].set_title('ROC-Kurve')
axes[1].set_title('Precision-Recall-Kurve')
plt.tight_layout(); plt.show()

**Wann welche Klassifikationsmetrik?**

- **Accuracy**: gut bei ungefähr gleich großen Klassen und ähnlichen Fehlerkosten.
- **Balanced Accuracy**: besser bei Klassenungleichgewicht.
- **Recall**: wichtig, wenn Verpassen teuer ist, z. B. medizinische Risiken.
- **Precision**: wichtig, wenn Fehlalarme teuer sind.
- **F1**: Kompromiss aus Precision und Recall.
- **ROC AUC**: allgemeine Trennfähigkeit über Thresholds.
- **PR AUC**: besonders hilfreich bei seltenen positiven Klassen.

### Aufgaben

1. Verschiebe den Threshold und beobachte False Positives und False Negatives.
2. Warum ist Accuracy bei der Imbalanced Demo irreführend?

## 4. Unüberwachtes Lernen

Beim Training werden keine Labels verwendet. Echte Labels nutzen wir nur nachträglich zur Analyse.

In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data
yw = wine.target
Xw_scaled = StandardScaler().fit_transform(Xw)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Xw_pca = pca.fit_transform(Xw_scaled)

cluster_models = {
    'KMeans': KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE),
    'Agglomerative': AgglomerativeClustering(n_clusters=3),
    'DBSCAN': DBSCAN(eps=2.2, min_samples=5),
}
rows=[]
for name, model in cluster_models.items():
    labels = model.fit_predict(Xw_scaled)
    mask = labels != -1
    sil = silhouette_score(Xw_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    rows.append({'Modell': name, 'Silhouette': sil, 'ARI': adjusted_rand_score(yw, labels), 'NMI': normalized_mutual_info_score(yw, labels), 'Cluster': len(set(labels))})
display(pd.DataFrame(rows))

plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=yw, cmap='tab10')
plt.title('PCA-Projektion: echte Labels nur zur Ansicht')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

In [ ]:
def kmeans_widget(clusteranzahl=3):
    labels = KMeans(n_clusters=clusteranzahl, n_init=10, random_state=RANDOM_STATE).fit_predict(Xw_scaled)
    print('Silhouette:', round(silhouette_score(Xw_scaled, labels), 3), '| ARI:', round(adjusted_rand_score(yw, labels), 3), '| NMI:', round(normalized_mutual_info_score(yw, labels), 3))
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.title(f'KMeans mit {clusteranzahl} Clustern')
    plt.show()
widgets.interact(kmeans_widget, clusteranzahl=widgets.IntSlider(value=3, min=2, max=8));

In [ ]:
def dbscan_widget(eps=2.2, min_samples=5):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xw_scaled)
    mask = labels != -1
    sil = silhouette_score(Xw_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    print('Cluster inklusive Rauschen:', sorted(set(labels)), '| Silhouette:', sil)
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.title('DBSCAN: eps und min_samples steuern Dichte')
    plt.show()
widgets.interact(dbscan_widget, eps=widgets.FloatSlider(value=2.2, min=0.5, max=5.0, step=0.1), min_samples=widgets.IntSlider(value=5, min=2, max=15));

### Aufgaben

1. Stelle KMeans absichtlich auf 2 oder 6 Cluster. Was passiert mit ARI/NMI?
2. Suche DBSCAN-Parameter, bei denen fast alles als Rauschen (`-1`) markiert wird.

## 5. Bias-Varianz-Tradeoff

- **Bias**: Modell ist zu einfach und verfehlt die Grundstruktur.
- **Varianz**: Modell reagiert zu stark auf einzelne Trainingspunkte.
- **Underfitting**: hoher Trainings- und Validierungsfehler.
- **Overfitting**: niedriger Trainingsfehler, aber hoher Validierungsfehler.
- **Generalisierung**: gute Leistung auf neuen Daten.

In [ ]:
def make_nonlinear(n=80, noise=0.25):
    rng = np.random.default_rng(RANDOM_STATE)
    X = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
    y = np.sin(X[:, 0]) + 0.35 * X[:, 0] + rng.normal(0, noise, n)
    return X, y

def bias_variance_demo(polynomgrad=3, rauschen=0.25, trainingsdaten=80):
    Xn, yn = make_nonlinear(trainingsdaten, rauschen)
    Xtr, Xval, ytr, yval = train_test_split(Xn, yn, test_size=0.35, random_state=RANDOM_STATE)
    model = make_pipeline(PolynomialFeatures(polynomgrad), LinearRegression())
    model.fit(Xtr, ytr)
    grid = np.linspace(-3, 3, 300).reshape(-1, 1)
    train_rmse = np.sqrt(mean_squared_error(ytr, model.predict(Xtr)))
    val_rmse = np.sqrt(mean_squared_error(yval, model.predict(Xval)))
    plt.scatter(Xtr, ytr, label='Training', alpha=0.8)
    plt.scatter(Xval, yval, label='Validierung', alpha=0.8)
    plt.plot(grid, model.predict(grid), color='black', label='Modell')
    plt.title(f'Grad {polynomgrad}: Train RMSE={train_rmse:.2f}, Val RMSE={val_rmse:.2f}')
    plt.legend(); plt.show()

widgets.interact(bias_variance_demo, polynomgrad=widgets.IntSlider(3, 1, 15), rauschen=widgets.FloatSlider(0.25, 0.0, 1.0, 0.05), trainingsdaten=widgets.IntSlider(80, 20, 200, 10));

In [ ]:
Xn, yn = make_nonlinear(160, 0.3)
model = make_pipeline(PolynomialFeatures(6), LinearRegression())
train_sizes, train_scores, val_scores = learning_curve(model, Xn, yn, cv=5, scoring='neg_root_mean_squared_error', train_sizes=np.linspace(0.2, 1.0, 5))
plt.plot(train_sizes, -train_scores.mean(axis=1), marker='o', label='Training')
plt.plot(train_sizes, -val_scores.mean(axis=1), marker='o', label='Validierung')
plt.xlabel('Anzahl Trainingsdaten')
plt.ylabel('RMSE')
plt.title('Learning Curve')
plt.legend(); plt.show()

### Aufgaben

1. Finde einen Polynomgrad, der sichtbar underfittet.
2. Erhöhe den Polynomgrad stark und reduziere die Trainingsdaten. Warum steigt die Varianz?

## 6. Metrikwahl in der Praxis

| Situation | Sinnvolle Metrik |
| --- | --- |
| Regression mit Ausreißern | MAE oder Median Absolute Error |
| Regression mit kritisch großen Fehlern | RMSE |
| Regression zur erklärten Varianz | R² |
| Klassifikation mit Klassenungleichgewicht | Balanced Accuracy, F1, PR AUC oder klassenspezifischer Recall |
| Medizinische Risikofälle | Recall für die positive Klasse |
| Hohe Fehlalarmkosten | Precision |

### Reflexionsaufgabe

Wähle ein eigenes Beispiel aus deinem Alltag. Welche Fehlerart wäre teurer: ein False Positive oder ein False Negative? Welche Metrik passt dazu?